In [1]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import  TextFormatter

import google.generativeai as genai

import chromadb
from chromadb.utils import embedding_functions

from chromadb.utils.embedding_functions import GoogleGeminiEmbeddingFunction

import os
from dotenv import load_dotenv

c:\Users\Anuj\Desktop\learn_chromadb\.venv\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\Anuj\Desktop\learn_chromadb\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Anuj\AppData\Local\Temp\ipykernel_28456\3630074394.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for m

In [5]:
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

In [6]:
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)

genai_model = genai.GenerativeModel('models/gemini-3.1-flash-lite')

chroma_client = chromadb.PersistentClient(path='chroma_db')

gemini_ef = embedding_functions.GoogleGeminiEmbeddingFunction(
    
    model_name="gemini-embedding-001",
    task_type="RETRIEVAL_DOCUMENT"
)


chroma_collection = chroma_client.get_or_create_collection(name='yt_notes', embedding_function=gemini_ef)

In [43]:
#input
#https://youtu.be/IdLSZEYlWVo
#https://youtu.be/vRKrkU-SoUQ&list=PLIveiH2sYUbi0TJfYKWwdkJBsnlABlSzb

yt_video_id='vRKrkU-SoUQ'

prompt = "Extract key notes from  video transcript: "

In [9]:
pip install youtube-transcript-api

Note: you may need to restart the kernel to use updated packages.


In [12]:
from youtube_transcript_api import YouTubeTranscriptApi

In [44]:
transcript = YouTubeTranscriptApi().fetch(yt_video_id,languages=['en','en-US','en-IN'])
transcript = TextFormatter().format_transcript(transcript)

with open(input("enter the name of the text file you want to save it with .txt extension"),"w",) as file:
    file.write(transcript)


In [45]:
#generate notes

response = genai_model.generate_content(prompt + transcript, stream=False)

with open(input("enter the name of your note with .txt extension"),"w") as file:
    file.write(response.text)

In [49]:
#storeing the notes in chromadb

with open(input("enter the name of your notes with .txt extension"),"r")as file:
    notes = file.read()
    
chroma_collection.upsert(
    documents=[notes],
    ids=[yt_video_id]
)

#validation

result = chroma_collection.get(yt_video_id, include=['documents'])
result

{'ids': ['vRKrkU-SoUQ'],
 'embeddings': None,
 'documents': ['Here are the key notes from the pizza masterclass transcript:\n\n### **Course Overview**\n*   **Program:** "Do It Yourself" (DIY) Pizza Making Masterclass.\n*   **Goal:** To teach home cooks how to craft authentic pizzas, from dough preparation to signature sauces.\n\n### **History of Pizza**\n*   **Origin:** Born in Naples, Italy, in the late 18th century as a quick meal for the local working class.\n*   **Etymology:** The word "pizza" was first recorded in 997 AD in Gaeta, Italy.\n*   **The Modern Pizza:** Credited to Raffaele Esposito, who created the "Margherita" pizza for Queen Margherita. He used basil (green), mozzarella (white), and tomatoes (red) to represent the colors of the Italian flag.\n*   **Global Spread:** In the 1940s, Italian immigrants brought their pizza traditions to America, sparking a food revolution that eventually turned pizza into a global phenomenon.\n\n### **The Modern Pizza Evolution**\n*   **Ma

In [ ]:
#search notes

query_text = input("enter your query")
n_results = 2

results = chroma_collection.query(
    query_texts=[query_text],
    n_results=n_results,
    include=['documents','metadatas','distances']
)


for i in range(len(results['ids'][0])):
    id = results["ids"][0][i]
    document = results['documents'][0][i]
    
    print("************************************************************************************")
    print(f'{i+1}. https://youtu.be/{id}')
    print("************************************************************************************")
    print(document)

************************************************************************************
1. https://youtu.be/vRKrkU-SoUQ
************************************************************************************
Here are the key notes from the pizza masterclass transcript:

### **Course Overview**
*   **Program:** "Do It Yourself" (DIY) Pizza Making Masterclass.
*   **Goal:** To teach home cooks how to craft authentic pizzas, from dough preparation to signature sauces.

### **History of Pizza**
*   **Origin:** Born in Naples, Italy, in the late 18th century as a quick meal for the local working class.
*   **Etymology:** The word "pizza" was first recorded in 997 AD in Gaeta, Italy.
*   **The Modern Pizza:** Credited to Raffaele Esposito, who created the "Margherita" pizza for Queen Margherita. He used basil (green), mozzarella (white), and tomatoes (red) to represent the colors of the Italian flag.
*   **Global Spread:** In the 1940s, Italian immigrants brought their pizza traditions to America,

In [ ]:
#generating answer from the notes based on the text query

prompt = "Answer flowwing QUESTIONS using DOCUMENT  as context."
prompt+=f"QUESTION: {query_text}"
prompt+=f"DOCUMENT: {results['documents'][0][0]}"

response = genai_model.generate_content(prompt, stream=False)
print(response.text)

Based on the document provided, here is the history of pizza:

*   **Etymology:** The word "pizza" was first recorded in 997 AD in Gaeta, Italy.
*   **Origin:** Pizza originated in Naples, Italy, in the late 18th century as a quick meal for the local working class.
*   **The Modern Pizza:** Raffaele Esposito is credited with creating the "Margherita" pizza for Queen Margherita. He used basil (green), mozzarella (white), and tomatoes (red) to represent the colors of the Italian flag.
*   **Global Spread:** In the 1940s, Italian immigrants brought their traditions to America, which sparked a food revolution and turned pizza into a global phenomenon.
